## Install required libraries

In [ ]:
!pip uninstall -y torch torchvision torchaudio torch-geometric pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv
!pip cache purge

!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
  --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!pip install torch_geometric
!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.6.0+cu124.html

## Import Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torch_geometric.nn import (
    GCNConv, GATConv, GATv2Conv, GINConv, 
    global_mean_pool, global_add_pool
)
from torch_geometric.data import Data
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R_scipy
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2
from natsort import natsorted
import gc

### Setting seed for environment

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

## KiTTi Intrinsics Load 

### Since Image 2 was used throughout - load P2 camera intrinsics

In [ ]:
def get_kitti_intrinsics(calib_file):
    if not os.path.exists(calib_file):
        # If calibration path missing
        return np.array([
            [718.856, 0, 607.1928],
            [0, 718.856, 185.2157],
            [0, 0, 1]
        ])
        
    with open(calib_file, 'r') as f:
        for line in f.readlines():
            if line.startswith('P2:'):
                values = [float(x) for x in line.strip().split()[1:]]
                P2 = np.array(values).reshape(3, 4)
                K = P2[:3, :3]
                return K
    return np.eye(3)

## Poses Transformation code

In [ ]:
def rotation_matrix_to_quaternion(R_batch):
    R_np = R_batch.cpu().numpy()
    quat_xyzw = R_scipy.from_matrix(R_np).as_quat()
    quat_wxyz = np.hstack((quat_xyzw[:, 3:], quat_xyzw[:, :3]))
    return torch.from_numpy(quat_wxyz).to(R_batch.device, dtype=R_batch.dtype)

def quaternion_to_rotation_matrix(q, device):
    batch_size = q.shape[0]
    q = q / (torch.norm(q, p=2, dim=1, keepdim=True) + 1e-8)
    w, x, y, z = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
    
    xx, yy, zz = x*x, y*y, z*z
    xy, xz, yz = x*y, x*z, y*z
    wx, wy, wz = w*x, w*y, w*z
    
    R = torch.zeros((batch_size, 3, 3), device=device)
    R[:, 0, 0] = 1 - 2*yy - 2*zz; R[:, 0, 1] = 2*xy - 2*wz; R[:, 0, 2] = 2*xz + 2*wy
    R[:, 1, 0] = 2*xy + 2*wz; R[:, 1, 1] = 1 - 2*xx - 2*zz; R[:, 1, 2] = 2*yz - 2*wx
    R[:, 2, 0] = 2*xz - 2*wy; R[:, 2, 1] = 2*yz + 2*wx; R[:, 2, 2] = 1 - 2*xx - 2*yy
    return R

def skew_symmetric(t, device):
    batch_size = t.shape[0]
    T = torch.zeros((batch_size, 3, 3), device=device)
    T[:, 0, 1] = -t[:, 2]; T[:, 0, 2] =  t[:, 1]
    T[:, 1, 0] =  t[:, 2]; T[:, 1, 2] = -t[:, 0]
    T[:, 2, 0] = -t[:, 1]; T[:, 2, 1] =  t[:, 0]
    return T

## The 8 point method and Sampson Loss

In [ ]:
def eight_point_algorithm(kpts, device):
    p1, p2 = kpts[:, 0:3], kpts[:, 3:6]
    A = torch.stack([
        p2[:, 0]*p1[:, 0], p2[:, 0]*p1[:, 1], p2[:, 0]*p1[:, 2],
        p2[:, 1]*p1[:, 0], p2[:, 1]*p1[:, 1], p2[:, 1]*p1[:, 2],
        p2[:, 2]*p1[:, 0], p2[:, 2]*p1[:, 1], p2[:, 2]*p1[:, 2]
    ], dim=-1)
    
    _, _, Vh = torch.linalg.svd(A)
    E_prov = Vh[-1].view(3, 3)
    U, _, Vh = torch.linalg.svd(E_prov)
    S_corrected = torch.diag(torch.tensor([1., 1., 0.], device=device))
    return U @ S_corrected @ Vh

def sampson_error(kpts, E):
    p1, p2 = kpts[:, 0:3], kpts[:, 3:6]
    Ep1 = (E @ p1.T).T
    Etp2 = (E.T @ p2.T).T
    numerator = (torch.sum(p2 * Ep1, dim=1))**2
    denominator = Ep1[:, 0]**2 + Ep1[:, 1]**2 + Etp2[:, 0]**2 + Etp2[:, 1]**2
    return numerator / (denominator + 1e-8)

## K-NN Graph

In [ ]:
def custom_knn_graph(x, k=6, loop=False):
    N = x.size(0)
    actual_k = min(k, N - 1) if not loop else min(k, N)
    if actual_k <= 0: return torch.empty((2, 0), dtype=torch.long, device=x.device)
    dist = torch.cdist(x, x)
    if not loop: dist.fill_diagonal_(float('inf'))
    _, knn_indices = torch.topk(dist, k=actual_k, largest=False, dim=1)
    target_nodes = torch.arange(N, device=x.device).repeat_interleave(actual_k)
    source_nodes = knn_indices.flatten()
    return torch.stack([source_nodes, target_nodes], dim=0)

## Building Graph for GNN

In [ ]:
def build_graph_for_batch(keypoints, device, k=6, tau=1e-4):
    num_nodes = keypoints.shape[0]
    coords_p1 = keypoints[:, 0:2] / (keypoints[:, 2:3] + 1e-8)
    edge_index = custom_knn_graph(coords_p1, k=k, loop=False).to(device)

    if edge_index.shape[1] == 0:
        return Data(x=keypoints, edge_index=edge_index)

    rand_indices = torch.randperm(num_nodes)[:min(num_nodes, 32)]
    E_prov = eight_point_algorithm(keypoints[rand_indices], device)

    errors = sampson_error(keypoints, E_prov)
    inlier_node_mask = errors < tau
    
    src, dst = edge_index
    edge_mask = inlier_node_mask[src] & inlier_node_mask[dst]
    pruned_edge_index = edge_index[:, edge_mask]

    return Data(x=keypoints, edge_index=pruned_edge_index)

def build_geometric_graphs(keypoints, K, device, k=8, tau=0.01):
    uv1 = keypoints[:, 0:2]
    uv2 = keypoints[:, 3:5]
    
    Kinv = torch.inverse(K).to(device)
    ones = torch.ones((uv1.shape[0], 1), device=device)
    
    kp1_h = torch.cat([uv1, ones], dim=1)
    kp2_h = torch.cat([uv2, ones], dim=1)
    
    x1 = (Kinv @ kp1_h.T).T 
    x2 = (Kinv @ kp2_h.T).T 
    
    if x1.shape[0] < 8: return None, None, None
    
    rand_idx = torch.randperm(x1.shape[0], device=device)[:min(64, x1.shape[0])]
    E_prov = eight_point_algorithm(torch.cat([x1[rand_idx], x2[rand_idx]], dim=1), device)
    
    s_err = sampson_error(torch.cat([x1, x2], dim=1), E_prov)
    
    mask = s_err < tau
    if mask.sum() < 8: return None, None, None
    
    uv1, uv2 = uv1[mask], uv2[mask]
    x1, x2 = x1[mask], x2[mask]
    s_err = s_err[mask]
    
    edge_index = custom_knn_graph(uv1, k=k, loop=False).to(device)
    
    cx, cy = K[0, 2], K[1, 2]
    
    def get_node_features(uv, x_norm):
        uv_c = uv - torch.tensor([cx, cy], device=device)
        r = torch.norm(uv_c, dim=1, keepdim=True)
        return torch.cat([x_norm, uv, uv_c, r], dim=1)

    feat1 = get_node_features(uv1, x1)
    feat2 = get_node_features(uv2, x2)
    
    g1 = Data(x=feat1, edge_index=edge_index)
    g2 = Data(x=feat2, edge_index=edge_index)
    
    return g1, g2, s_err

## Defining the MLP decision head

In [ ]:
class BasePoseHead(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.pool = global_mean_pool
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        
        self.rot_head = nn.Linear(hidden_dim // 2, 4)
        self.trans_head = nn.Linear(hidden_dim // 2, 3)
        self.scale_head = nn.Linear(hidden_dim // 2, 1)

    def forward_heads(self, x, batch):
        gap_vector = self.pool(x, batch)          
        mlp1_out = F.relu(self.fc1(gap_vector))
        x_global = F.relu(self.fc2(mlp1_out))
        
        pred_q = self.rot_head(x_global)        
        pred_t_raw = self.trans_head(x_global)  
        pred_scale = F.softplus(self.scale_head(x_global)) 
        
        pred_q = pred_q / (torch.norm(pred_q, p=2, dim=1, keepdim=True) + 1e-8)
        pred_t_norm = pred_t_raw / (torch.norm(pred_t_raw, p=2, dim=1, keepdim=True) + 1e-8)
        
        return pred_q, pred_t_norm, pred_t_raw, pred_scale

## Defining the GNN Models

In [ ]:
class GNNPose_3GCN_then_GAT(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=128, num_heads=4):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.conv_gat = GATConv(hidden_dim, hidden_dim, heads=num_heads, concat=False)
        self.head = BasePoseHead(hidden_dim)

    def forward(self, x, edge_index, batch, return_analysis=False):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = self.conv3(x, edge_index).relu()
        
        if return_analysis:
            x_gat, gat_weights = self.conv_gat(x, edge_index, return_attention_weights=True)
            x_gat = x_gat.relu()
            pred_q, pred_t_norm, pred_t_raw, pred_scale = self.head.forward_heads(x_gat, batch)
            return pred_q, pred_t_norm, pred_t_raw, gat_weights[1], pred_scale
        else:
            x = self.conv_gat(x, edge_index).relu()
            return self.head.forward_heads(x, batch)

class GNNPose_GAT_then_2GCN(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=128, num_heads=4):
        super().__init__()
        self.conv_gat = GATConv(input_dim, hidden_dim, heads=num_heads, concat=False)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.head = BasePoseHead(hidden_dim)

    def forward(self, x, edge_index, batch):
        x = self.conv_gat(x, edge_index).relu()
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        return self.head.forward_heads(x, batch)

def mlp(in_dim, out_dim, hidden_dim=None):
    if hidden_dim is None: hidden_dim = out_dim
    return nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, out_dim))

class PoseGIN_SumPool(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=128):
        super().__init__()
        self.mlp1 = mlp(input_dim, hidden_dim, hidden_dim)
        self.gin1 = GINConv(self.mlp1, train_eps=True)
        self.mlp2 = mlp(hidden_dim, hidden_dim, hidden_dim)
        self.gin2 = GINConv(self.mlp2, train_eps=True)
        
        self.fc_shared = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
        self.rot_head = nn.Linear(hidden_dim, 4)
        self.trans_head = nn.Linear(hidden_dim, 3)
        self.scale_head = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, batch):
        x_emb = F.relu(self.gin1(x, edge_index))
        x_emb = F.relu(self.gin2(x_emb, edge_index))
        x_pool = global_add_pool(x_emb, batch)
        
        x_dense = self.fc_shared(x_pool)
        
        pred_q = F.normalize(self.rot_head(x_dense), p=2, dim=1)
        pred_t_raw = self.trans_head(x_dense)
        pred_t_norm = F.normalize(pred_t_raw, p=2, dim=1)
        pred_scale = F.softplus(self.scale_head(x_dense))
        
        return pred_q, pred_t_norm, pred_t_raw, pred_scale

class CrossGraphAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.multihead_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.ReLU(), nn.Linear(embed_dim, embed_dim))

    def forward(self, x1, x2):
        q, k, v = x1.unsqueeze(0), x2.unsqueeze(0), x2.unsqueeze(0)
        attn_out, _ = self.multihead_attn(q, k, v)
        x = self.norm(x1 + attn_out.squeeze(0))
        return x + self.ffn(x)

class GeometricGraphPoseNet(nn.Module):
    def __init__(self, node_dim=8, hidden_dim=128):
        super().__init__()
        self.gat1 = GATv2Conv(node_dim, hidden_dim // 2, heads=2, concat=True)
        self.gat2 = GATv2Conv(hidden_dim, hidden_dim, heads=1, concat=False)
        self.cross_attn = CrossGraphAttention(hidden_dim)
        
        self.motion_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 3, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim)
        )
        self.pool_gate = nn.Linear(hidden_dim + 1, 1)
        self.rot_head = nn.Linear(hidden_dim, 4)
        self.trans_dir_head = nn.Linear(hidden_dim, 3) 
        self.scale_head = nn.Linear(hidden_dim, 1)

    def forward(self, g1, g2, sampson_err):
        h1 = self.gat2(F.elu(self.gat1(g1.x, g1.edge_index)), g1.edge_index)
        h2 = self.gat2(F.elu(self.gat1(g2.x, g2.edge_index)), g2.edge_index)
        
        h_cross = self.cross_attn(h1, h2)
        uv1, uv2 = g1.x[:, 3:5], g2.x[:, 3:5]
        
        flow_mag = torch.norm(uv2 - uv1, dim=1, keepdim=True)
        center = torch.mean(uv1, dim=0, keepdim=True)
        radial_flow = torch.norm(uv2 - center, dim=1, keepdim=True) - torch.norm(uv1 - center, dim=1, keepdim=True)

        combined = torch.cat([h1, h_cross, flow_mag, radial_flow, sampson_err.unsqueeze(1)], dim=1)
        feat = self.motion_mlp(combined)
        
        weights = torch.sigmoid(self.pool_gate(torch.cat([feat, sampson_err.unsqueeze(1)], dim=1)))
        global_feat = (torch.sum(feat * weights, dim=0) / (torch.sum(weights) + 1e-6)).unsqueeze(0)
        
        pred_q = F.normalize(self.rot_head(global_feat), dim=1)
        pred_t_dir = F.normalize(self.trans_dir_head(global_feat), dim=1)
        pred_scale = F.softplus(self.scale_head(global_feat))
        
        pred_t_final = pred_t_dir * pred_scale
        
        return pred_q, pred_t_dir, pred_t_final, pred_scale

## Loss Function for 3 GNN Modules

In [ ]:
class UnifiedCompositeLoss(nn.Module):
    def __init__(self, use_scale=False, use_essential=False, use_yaw=False, use_weighted=False):
        super().__init__()
        self.use_scale, self.use_essential, self.use_yaw = use_scale, use_essential, use_yaw
        self.w_t = [5.0, 5.0, 10.0] if use_weighted else [1.0, 1.0, 1.0]
        self.w_q = [10.0, 10.0, 10.0, 10.0] if use_weighted else [1.0, 1.0, 1.0, 1.0]
        self.w_heading, self.gamma_E, self.lambda_scale = (5.0 if use_weighted else 1.0), 1.0, 0.2
        self.mse = nn.MSELoss()
        self.l1 = nn.L1Loss()

    def get_heading(self, q):
        x, y, z, w = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
        return torch.atan2(2*(w*y - z*x), 1 - 2*(y*y + z*z))

    def forward(self, pred_q, pred_t, pred_scale, gt_q, gt_t, gt_scale, gt_E, device):
        dot = torch.sum(pred_q * gt_q, dim=1, keepdim=True)
        pred_q = torch.where(dot < 0, -pred_q, pred_q)

        loss = sum([w*self.mse(pred_q[:,i], gt_q[:,i]) for i,w in enumerate(self.w_q)]) + \
               sum([w*self.mse(pred_t[:,i], gt_t[:,i]) for i,w in enumerate(self.w_t)])

        if self.use_scale:
            loss += self.lambda_scale * self.l1(pred_scale, gt_scale)

        if self.use_essential:
            R_pred = quaternion_to_rotation_matrix(pred_q, device)
            E_pred = torch.bmm(skew_symmetric(pred_t, device), R_pred)
            loss_E_frob = torch.mean(torch.norm(E_pred - gt_E, p='fro', dim=(1, 2)))
            try:
                _, S, _ = torch.linalg.svd(E_pred)
                loss_E_svd = torch.mean((S[:, 0] - S[:, 1])**2 + S[:, 2]**2)
            except RuntimeError:
                loss_E_svd = torch.tensor(0.0, device=device)
            loss += self.gamma_E * (loss_E_frob + loss_E_svd)

        if self.use_yaw:
            loss += self.w_heading * torch.mean(1 - torch.cos(self.get_heading(pred_q) - self.get_heading(gt_q)))

        return loss

## Loss Function for the 4th GNN Module

In [ ]:
class GeometricConsistencyLoss(nn.Module):
    def __init__(self, w_q=10.0, w_t_dir=5.0, w_scale=1.0, w_E=1.0):
        super().__init__()
        self.w_q, self.w_t_dir, self.w_scale, self.w_E = w_q, w_t_dir, w_scale, w_E
        self.l1 = nn.L1Loss()
        
    def forward(self, pred_q, pred_t, pred_scale, gt_q, gt_t, gt_scale, gt_E, device):
        dot = torch.sum(pred_q * gt_q, dim=1, keepdim=True)
        pred_q_aligned = torch.where(dot < 0, -pred_q, pred_q)
        loss_q = self.l1(pred_q_aligned, gt_q)
        
        loss_t_dir = 1.0 - F.cosine_similarity(pred_t, gt_t).mean()
        loss_scale = self.l1(pred_scale, gt_scale)
        
        R = quaternion_to_rotation_matrix(pred_q, device=device)
        T_skew = skew_symmetric(pred_t, device=device)
        E_pred = T_skew @ R
        
        E_pred_norm = E_pred / (torch.norm(E_pred, dim=(1,2), keepdim=True) + 1e-8)
        gt_E_norm = gt_E / (torch.norm(gt_E, dim=(1,2), keepdim=True) + 1e-8)
        loss_E = torch.mean(torch.norm(E_pred_norm - gt_E_norm, p='fro', dim=(1,2)))
        
        return (self.w_q * loss_q) + (self.w_t_dir * loss_t_dir) + (self.w_scale * loss_scale) + (self.w_E * loss_E)

## Dataset Class

In [ ]:
class CameraPoseDataset(Dataset):
    def __init__(self, keypoints, rotations, translations):
        self.keypoints, self.gt_R, self.gt_t_raw = keypoints, rotations, translations
        self.num_samples = self.keypoints.shape[0]

        if self.gt_t_raw.dim() != 3 or self.gt_t_raw.shape[2] != 1:
            self.gt_t_raw = self.gt_t_raw.view(self.num_samples, 3, 1)
        
        self.gt_scale = torch.norm(self.gt_t_raw, dim=1, keepdim=True)
        self.gt_t = self.gt_t_raw / (self.gt_scale + 1e-8)
        
        self.gt_q = rotation_matrix_to_quaternion(self.gt_R)
        t_skew_batch = skew_symmetric(self.gt_t.squeeze(-1), 'cpu')
        self.gt_E = torch.bmm(t_skew_batch, self.gt_R)

    def __len__(self): return self.num_samples

    def __getitem__(self, idx):
        return {
            'keypoints': self.keypoints[idx], 
            'gt_E': self.gt_E[idx],
            'gt_q': self.gt_q[idx], 
            'gt_t': self.gt_t[idx].squeeze(-1),
            'gt_scale': self.gt_scale[idx].squeeze(-1) 
        }

## Function to Train the model

### Update to your model saved paths

In [ ]:
def train_eval_model(model_name, model, train_loader, val_loader, criterion, device, K_intrinsics=None):
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    save_path = f"{model_name}_ind_v1_best.pth"
    
    pretrained_paths = { # Update to model saved paths
        "3GCN_GAT": "model-path",
        "GAT_2GCN": "model-path",
        "CrossGraph": "model-path",
        "GIN_SumPool": "model-path"
    }
    
    load_path = pretrained_paths.get(model_name, "")
    
    if os.path.exists(load_path):
        try:
            model.load_state_dict(torch.load(load_path, map_location=device))
            print(f"Loaded existing pretrained checkpoint: {load_path}")
        except Exception as e:
            print(f"Pretrained checkpoint found but failed to load ({e}). Training from scratch.")
    else:
        print(f"No pretrained checkpoint found at {load_path}. Training {model_name} from scratch.")

    best_val_loss = float('inf')
    patience_counter = 0
    max_epochs = 20
    patience = 8
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            kpts = batch['keypoints'].squeeze(0).to(device)
            gt_q = batch['gt_q'].to(device)
            gt_t = batch['gt_t'].to(device)
            gt_scale = batch['gt_scale'].to(device)
            gt_E = batch['gt_E'].to(device)
            
            optimizer.zero_grad()
            
            if model_name == "CrossGraph":
                g1, g2, s_err = build_geometric_graphs(kpts, K_intrinsics, device)
                if g1 is None: continue
                pred_q, pred_t, pred_t_raw, pred_scale = model(g1, g2, s_err)
            else:
                graph = build_graph_for_batch(kpts, device)
                if graph.num_nodes == 0 or graph.edge_index.shape[1] == 0: continue
                batch_indices = torch.zeros(graph.num_nodes, dtype=torch.long, device=device)
                pred_q, pred_t, pred_t_raw, pred_scale = model(graph.x, graph.edge_index, batch_indices)
                
            loss = criterion(pred_q, pred_t, pred_scale, gt_q, gt_t, gt_scale, gt_E, device)
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                kpts = batch['keypoints'].squeeze(0).to(device)
                gt_q = batch['gt_q'].to(device)
                gt_t = batch['gt_t'].to(device)
                gt_scale = batch['gt_scale'].to(device)
                gt_E = batch['gt_E'].to(device)
                
                if model_name == "CrossGraph":
                    g1, g2, s_err = build_geometric_graphs(kpts, K_intrinsics, device)
                    if g1 is None: continue
                    pred_q, pred_t, pred_t_raw, pred_scale = model(g1, g2, s_err)
                else:
                    graph = build_graph_for_batch(kpts, device)
                    if graph.num_nodes == 0 or graph.edge_index.shape[1] == 0: continue
                    batch_indices = torch.zeros(graph.num_nodes, dtype=torch.long, device=device)
                    pred_q, pred_t, pred_t_raw, pred_scale = model(graph.x, graph.edge_index, batch_indices)
                        
                loss = criterion(pred_q, pred_t, pred_scale, gt_q, gt_t, gt_scale, gt_E, device)
                val_loss += loss.item()
                
        avg_train = train_loss / max(1, len(train_loader))
        avg_val = val_loss / max(1, len(val_loader))
        
        print(f"[{model_name}] Epoch {epoch+1:02d} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
        
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print(f"  -> Best model saved! ({save_path})")
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f"[{model_name}] Early stopping triggered at epoch {epoch+1}")
            break
            
    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, map_location=device))
    return model

## Main code

### Update to your Dataset paths

In [ ]:
if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    KITTI_CALIB = 'your-calib-file'
    KITTI_IMGS = 'kitti-image2-seq-path'
    KPTS_PATH = 'processed-kpts-npy-path'
    ROT_PATH = 'processed-rot-arr-npy-path'
    TRAN_PATH = 'processed-tran-vec-npy-path'
    
    print("Loading KITTI Camera Intrinsics...")
    K_np = get_kitti_intrinsics(KITTI_CALIB)
    K_intrinsics = torch.tensor(K_np, dtype=torch.float32, device=device)
    
    print("Loading Preprocessed KITTI .npy arrays...")
    if os.path.exists(KPTS_PATH) and os.path.exists(ROT_PATH) and os.path.exists(TRAN_PATH):
        kpts_arr = np.load(KPTS_PATH)
        rot_arr = np.load(ROT_PATH)
        tran_arr = np.load(TRAN_PATH)
        
        dataset = CameraPoseDataset(
            torch.from_numpy(kpts_arr).float(),
            torch.from_numpy(rot_arr).float(),
            torch.from_numpy(tran_arr).float()
        )
    else:
        print("KITTI .npy files not found at specified paths. Generating robust mock data...")
        num_samples = 900 
        N_fixed = 100 # N is fixed but could be any value
        
        kpts_mock = torch.rand(num_samples, N_fixed, 6)
        rot_mock = torch.eye(3).unsqueeze(0).repeat(num_samples, 1, 1)
        trans_mock = torch.rand(num_samples, 3, 1)
        
        dataset = CameraPoseDataset(kpts_mock, rot_mock, trans_mock)
        
    # Train / Val Split (0.8 : 0.2)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
    
    models = {
        "3GCN_GAT": GNNPose_3GCN_then_GAT().to(device),
        "GAT_2GCN": GNNPose_GAT_then_2GCN().to(device),
        "GIN_SumPool": PoseGIN_SumPool().to(device),
        "CrossGraph": GeometricGraphPoseNet().to(device)
    }
    
    # L5 Unified Loss for first three models
    l5_loss = UnifiedCompositeLoss(use_scale=True, use_essential=True, use_yaw=True, use_weighted=True)
    # Geometric Consistency Loss specifically for CrossGraph model
    geo_loss = GeometricConsistencyLoss()

    print(f"Starting Training Routine (Total Valid Samples: {len(dataset)} | Train: {train_size} | Val: {val_size})\n" + "="*80)
    
    for name, model in models.items():
        print(f"\n--- Processing Model: {name} ---")
        criterion = geo_loss if name == "CrossGraph" else l5_loss
        models[name] = train_eval_model(name, model, train_loader, val_loader, criterion, device, K_intrinsics)
        
    print("\nAll models processed and best checkpoints updated.")